# Notebook 3 — FLAN-T5-XL: Financial Domain Prompt

**Model:** `google/flan-t5-xl` (~3B parameters, ~12GB in float32 / ~6GB in float16)  
**Task:** Text simplification using the same financial domain-specific prompt as Notebook 2  
**Input:** `data/final/dataset_model_v2.csv` (must contain `output_generic` and `output_financial`)  
**Output column:** `output_financial_t5xl`  
**Saved to:** `data/final/dataset_model_v2.csv`

---
This notebook tests whether **scaling up the model** (base → XL, 250M → 3B parameters)  
improves simplification quality while keeping the prompt identical to Notebook 2.

> ⚠️ **Memory note:** This model requires ~6GB RAM in float16. The code uses `device_map="auto"` and `torch_dtype=torch.float16` to manage memory automatically.

## 0. Environment Info

In [ ]:
# Environment info — run this first to log versions for reproducibility
import platform, importlib
for pkg in ["torch", "transformers", "sentencepiece"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:<15}: {mod.__version__}")
    except Exception:
        print(f"{pkg:<15}: not installed")
print(f"{'python':<15}: {platform.python_version()}")
print(f"{'platform':<15}: {platform.platform()}")

## 1. Install Dependencies

In [ ]:
import subprocess, sys

packages = ['transformers==4.40.0', 'torch==2.2.0', 'sentencepiece==0.1.99', 'accelerate==0.29.0']
for pkg in packages:
    base = pkg.split('==')[0].replace('-', '_')
    try:
        __import__(base)
        print(f'  {pkg} already installed')
    except ImportError:
        print(f'  Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  {pkg} installed')

## 2. Imports

In [ ]:
import pandas as pd
import torch
import re
import psutil
from transformers import T5ForConditionalGeneration, T5Tokenizer

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

mem = psutil.virtual_memory()
print(f"Total RAM       : {mem.total / 1e9:.1f} GB")
print(f"Available RAM   : {mem.available / 1e9:.1f} GB")

## 2b. Set Random Seed

In [ ]:
import torch, random, numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f'Random seed set to {SEED}')

## 3. Load Dataset

In [ ]:
DATA_PATH = "../data/final/dataset_model_v2.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")

assert "output_generic" in df.columns,   "Run Notebook 1 first!"
assert "output_financial" in df.columns, "Run Notebook 2 first!"

df[["candidate_id", "text"]].head(3)

## 4. Load FLAN-T5-XL Model

Uses `float16` to halve memory usage (~6GB instead of ~12GB).  
`device_map="auto"` lets HuggingFace Accelerate distribute the model across available devices automatically.

In [ ]:
model_name = "google/flan-t5-xl"

print(f"Loading tokenizer: {model_name}")
tokenizer = T5Tokenizer.from_pretrained(model_name)

print(f"Loading model: {model_name} (float16, ~6GB)...")
model = T5ForConditionalGeneration.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()
print("Model loaded.")

mem = psutil.virtual_memory()
print(f"RAM available after load: {mem.available / 1e9:.1f} GB")

## 5. Define Prompt

> Same financial domain prompt as Notebook 2 — only the model changes.

In [ ]:
PROMPT_PREFIX = (
    "You are a financial document simplification expert. "
    "Rewrite the following financial or regulatory text in plain English for a non-expert consumer. "
    "Keep all numbers, dates, and legal conditions exactly as written. "
    "Do not add any information not in the original. "
    "Preserve the original meaning exactly:\n\n"
)

example_prompt = PROMPT_PREFIX + str(df["text"].iloc[0])
print("=== Example Prompt ===")
print(example_prompt)
print(f"\nToken count: {len(tokenizer(example_prompt)['input_ids'])}")

## 6. Run Inference

In [ ]:
outputs = []
total = len(df)

for i, row in df.iterrows():
    prompt = PROMPT_PREFIX + str(row["text"])
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=4,
            early_stopping=True
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    outputs.append(decoded)

    row_num = i + 1
    if row_num % 10 == 0:
        print(f"Processed {row_num}/{total} rows...")

print(f"Processed {total}/{total} rows. Done.")

## 7. Save Results

In [ ]:
df["output_financial_t5xl"] = outputs
df.to_csv(DATA_PATH, index=False)
print(f"Saved to {DATA_PATH}")
print(f"Columns: {df.columns.tolist()}")

## 8. Summary Statistics

In [ ]:
def approx_fkgl(text):
    words = str(text).split()
    sentences = [s.strip() for s in re.split(r'[.!?]+', str(text)) if s.strip()]
    num_sentences = max(len(sentences), 1)
    num_words = max(len(words), 1)
    num_syllables = sum(max(1, len(w) // 3) for w in words)
    return round(0.39 * (num_words / num_sentences) + 11.8 * (num_syllables / num_words) - 15.59, 2)

df["wc_t5xl"]   = df["output_financial_t5xl"].apply(lambda x: len(str(x).split()))
df["fkgl_t5xl"] = df["output_financial_t5xl"].apply(approx_fkgl)
identical = (df["output_financial_t5xl"].str.strip() == df["text"].str.strip()).sum()

print("=" * 50)
print("SUMMARY — FLAN-T5-XL Financial Domain Prompt")
print("=" * 50)
print(f"Total rows processed        : {len(df)}")
print(f"Avg word count (output)     : {df['wc_t5xl'].mean():.2f}")
print(f"Avg FKGL (output)           : {df['fkgl_t5xl'].mean():.2f}")
print(f"Rows identical to input     : {identical}/{len(df)}")

## 9. Full Three-Model Comparison

In [ ]:
rows = [
    ("text (original)",          "text"),
    ("text_simplified_reference","text_simplified_reference"),
    ("output_generic",           "output_generic"),
    ("output_financial",         "output_financial"),
    ("output_financial_t5xl",    "output_financial_t5xl"),
]

results = []
for label, col in rows:
    wc   = df[col].apply(lambda x: len(str(x).split())).mean()
    fkgl = df[col].apply(approx_fkgl).mean()
    if col in ["text", "text_simplified_reference"]:
        ident = "—"
    else:
        ident = str((df[col].str.strip() == df["text"].str.strip()).sum())
    results.append({"Column": label, "Avg WC": round(wc, 2), "Avg FKGL": round(fkgl, 2), "Identical to Input": ident})

comparison = pd.DataFrame(results)
print(comparison.to_string(index=False))

## 10. Sample Output Comparison (All Three Models)

In [ ]:
# Show rows where T5-XL actually changed the text
changed = df[df["output_financial_t5xl"].str.strip() != df["text"].str.strip()].head(5)

for _, row in changed.iterrows():
    print(f"[Row {row['candidate_id']}]")
    print(f"  ORIGINAL  : {str(row['text'])[:120]}")
    print(f"  GENERIC   : {str(row['output_generic'])[:120]}")
    print(f"  FINANCIAL : {str(row['output_financial'])[:120]}")
    print(f"  T5-XL     : {str(row['output_financial_t5xl'])[:120]}")
    print(f"  REFERENCE : {str(row['text_simplified_reference'])[:120]}")
    print()

## 11. What Did the Model Actually Change?

Inspect the nature of changes: is it deleting words, replacing jargon, or just minor edits?

In [ ]:
changed_mask = df["output_financial_t5xl"].str.strip() != df["text"].str.strip()
df_changed   = df[changed_mask].copy()

df_changed["orig_wc"]   = df_changed["text"].apply(lambda x: len(str(x).split()))
df_changed["output_wc"] = df_changed["output_financial_t5xl"].apply(lambda x: len(str(x).split()))
df_changed["wc_delta"]  = df_changed["output_wc"] - df_changed["orig_wc"]

print(f"Rows with actual changes: {len(df_changed)}/100")
print(f"Average word count change: {df_changed['wc_delta'].mean():.2f} words")
print(f"Rows where output is SHORTER: {(df_changed['wc_delta'] < 0).sum()}")
print(f"Rows where output is LONGER : {(df_changed['wc_delta'] > 0).sum()}")
print(f"Rows same length but different: {(df_changed['wc_delta'] == 0).sum()}")

---
## Summary & Conclusions

| Model | Prompt | Avg WC | Avg FKGL | Identical to Input |
|---|---|---|---|---|
| Original | — | 41.82 | 11.61 | — |
| Reference (Claude API) | — | 40.94 | 10.82 | — |
| FLAN-T5-base | Generic | 30.62 | 11.68 | 15/100 |
| FLAN-T5-base | Financial | 36.25 | 11.77 | 36/100 |
| **FLAN-T5-XL** | Financial | **37.02** | **11.55** | **23/100** |

### Key takeaways

1. **Larger model ≠ solved problem.** XL reduces identical-to-input from 36→23, but still cannot produce meaningful simplification.
2. **FKGL barely moves.** All three models produce output at roughly the same reading level as the original (FKGL ~11.5–11.8). None approach the reference FKGL of 10.82.
3. **T5 architecture is the bottleneck.** T5 models are encoder-decoder and trained for QA/translation. They lack the generative fluency needed for true rewriting. Decoder-only models (LLaMA, Mistral, GPT) handle this task far better.
4. **The reference itself is not human-annotated.** The `text_simplified_reference` column was generated by Claude API, not human experts. This means the "gold standard" is also imperfect — and the small FKGL difference (11.61→10.82) reflects Claude's conservative approach under similar constraints.
5. **Next steps:** Fine-tune a decoder-only model (e.g. LLaMA 3.2-3B) on the 100 pairs in this dataset, or use Claude/GPT-4o API with a revised prompt that explicitly targets lower FKGL.